# Racecar — from LQR to MPC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro501/racecar_toward_mpc.ipynb)

LQR solves an infinite-horizon quadratic problem on a linear model. The algebraic Riccati equation gives one gain $K$, and the law $u = \bar u - K\,(x - \bar x)$ is applied for all time.

On a circuit the cost is no longer a quadratic form: stay in the lane, miss the cones, and keep $u$ inside a box $\mathcal{U}$. The prediction $\dot x = f(x,u)$ may also stay nonlinear. The finite-horizon program below is what a numerical solver is asked to solve at each time $t$. Resolving it every $\Delta t$ and applying only the first input is **model-predictive control (MPC)**.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.


In [ ]:
# Local: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


In [ ]:
import numpy as np

from minilink import (
    PlanningProblem,
    QuadraticCost,
    TrajectoryOptimizationPlanner,
    UdeSRacecar,
)
from minilink.control.mpc import ModelPredictiveController, mpc_animation_overlays
from minilink.graphical.catalog.racecar_skin import racecar_skin_2d, racecar_skin_3d
from minilink.planning import (
    ReferenceTrack,
    Scene,
    Sphere,
    bind,
    circuit_waypoints,
    from_waypoints,
    point_probe,
    quadratic_hinge,
)

V_REF = 3.5  # [m/s] cruise the quadratic term asks for
TF = 10.0
MPC_DT = 0.1
MPC_HORIZON = 1.0
N_STEPS = int(MPC_HORIZON / MPC_DT)  # number of steps in the prediction


## 1. From LQR to a finite-horizon program

The autopilot LQR is the infinite-horizon linear-quadratic program

$$
\min_{u(\cdot)}
\int_0^{\infty}
\Big(
\tilde x(\tau)^\top Q\,\tilde x(\tau)
+
\tilde u(\tau)^\top R\,\tilde u(\tau)
\Big)\,d\tau
\qquad
\text{s.t.}
\quad
\dot{\tilde x} = A\tilde x + B\tilde u,
$$

with $\tilde x = x - \bar x$ and $\tilde u = u - \bar u$. The optimum is the linear feedback $u = \bar u - K\tilde x$, with $K$ computed once by the Riccati equation.

A lap on a circuit asks for a different program:

1. The running cost $g$ is not quadratic in $x$: zero in the lane, growing outside it; zero far from a cone, growing near it.
2. The input must lie in a box $\mathcal{U}$ (speed and steer limits).
3. The prediction $\dot x = f(x,u)$ may be nonlinear.

At time $t$, given the measured state $x(t)$, the finite-horizon problem is

$$
\begin{aligned}
\min_{x(\cdot),\,u(\cdot)}
\quad
&
\int_{t}^{t+T} g\big(x(\tau),\,u(\tau)\big)\,d\tau
+
h\big(x(t+T)\big)
\\
\text{s.t.}
\quad
&
\dot x(\tau) = f\big(x(\tau),\,u(\tau)\big),
\\
&
x(t) = x_{\mathrm{measured}},
\\
&
u(\tau)\in\mathcal{U}.
\end{aligned}
$$

$T$ is the horizon (one second below), $g$ is the running cost, and $h \equiv 0$. The set $\mathcal{U}$ is the actuator box. The lane and the cones enter $g$ as soft penalties.

**MPC** resolves that same program from the new $x(t)$ every $\Delta t$, applies the first input, and discards the rest of the plan.


## 2. Circuit, lane, and cones

A rounded rectangle is the reference path. `ReferenceTrack` is the lane of half-width $0.6\,\mathrm{m}$. `Scene` holds the cones. Both enter the running cost $g$.


In [ ]:
path = circuit_waypoints(length=6.0, width=4.0, radius=1.0)
track = ReferenceTrack(from_waypoints(path), half_width=0.6)
scene = Scene(
    obstacles=[
        Sphere((0.0, 2.0), 0.15),
        Sphere((-2.2, 2.3), 0.30),
    ]
)


In [ ]:
fig, ax = scene.plot(show=False, show_density=False)
track.plot(ax=ax)


## 3. Planning model

The planning model is the kinematic bicycle `UdeSRacecar`: state $x = [x,\, y,\, \theta]$, input $u = [v,\, \delta]$ (speed and steer). The bounds on $u$ are the set $\mathcal{U}$. The initial state $x_0$ is a little off the centerline.


In [ ]:
car = UdeSRacecar()
car.inputs["u"].lower_bound = np.array([0.0, -0.52])
car.inputs["u"].upper_bound = np.array([5.0, 0.52])
car.skin = racecar_skin_2d
car.camera_follow_frame = None
car.camera_scale = 4.0

start = path[0]
heading = np.arctan2(path[1, 1] - start[1], path[1, 0] - start[0])
x0 = np.array(
    [start[0] - 0.1 * np.sin(heading), start[1] + 0.1 * np.cos(heading), heading]
)
car.x0 = x0


## 4. Running cost

Three terms. A probe reads the position $(x,y)$ so the lane margin and the cone clearance are scalar fields of the state.

**Cruise.** $Q = 0$. The input weight asks for a cruise speed and a small steer:

$$
g_{\mathrm{quad}}(u)
=
\big(u - \bar u\big)^\top R\,\big(u - \bar u\big),
\qquad
\bar u = (v_{\mathrm{ref}},\, 0),
\quad
R = \mathrm{diag}(2,\,1).
$$

**Lane (one-sided hinge).** Let $m(x)$ be the signed margin to the corridor wall: $m\ge 0$ inside the lane, $m<0$ outside. A quadratic hinge is zero in the lane and quadratic once the car leaves it,

$$
g_{\mathrm{lane}}(x)
=
w_c\,\big(\max(-m(x),\, 0)\big)^2.
$$

**Cones (one-sided hinge on clearance).** Let $d(x)$ be the clearance to the nearest cone. The same hinge, with a $0.2\,\mathrm{m}$ buffer, is

$$
g_{\mathrm{obs}}(x)
=
w_o\,\big(\max(0.2 - d(x),\, 0)\big)^2.
$$

The running cost of the program is the sum

$$
g(x,u) = g_{\mathrm{quad}}(u) + g_{\mathrm{lane}}(x) + g_{\mathrm{obs}}(x).
$$

The terminal cost is $h\equiv 0$.

The figure is the spatial part $g_{\mathrm{lane}}+g_{\mathrm{obs}}$ on the plane (blue is cheap; red is the hinge outside the lane and around the cones). The cruise term $g_{\mathrm{quad}}$ depends only on $u$ and does not appear.


In [ ]:
probe = bind(car, point_probe())
quad = QuadraticCost.from_system(
    car, Q=np.zeros((3, 3)), R=np.diag([2.0, 1.0]), ubar=np.array([V_REF, 0.0])
)
corridor = track.corridor_field(probe).as_cost(weight=20.0, shaping=quadratic_hinge())
obstacle = scene.clearance_field(probe).as_cost(
    weight=40.0, shaping=quadratic_hinge(threshold=0.2)
)
cost = quad + corridor + obstacle


In [ ]:
from minilink.planning.spatial.grid import sample_field_costs
from minilink.planning.spatial.plotting import plot_cost_field

pad = 1.5
bounds = (
    (float(path[:, 0].min()) - pad, float(path[:, 0].max()) + pad),
    (float(path[:, 1].min()) - pad, float(path[:, 1].max()) + pad),
)
field = sample_field_costs(
    [corridor, obstacle],
    bounds=bounds,
    state_dim=car.n,
    grid=(120, 120),
    u=np.zeros(car.m),
)
fig, ax = plot_cost_field(
    field, show=False, log_scale=True, title=r"$g_{\mathrm{lane}} + g_{\mathrm{obs}}$"
)
scene.plot(show=False, ax=ax, bounds=bounds, show_density=False, title="")
track.plot(show=False, ax=ax, bounds=bounds, title="")


## 5. Planning problem

The finite-horizon program of §1 is specified by the ingredients below.

**Planning model.** A prediction $\dot x = f(x,u)$ that ties the planned state to the planned input. Here $f$ is the kinematic bicycle: $x = [x,\, y,\, \theta]$ (position and heading) and $u = [v,\, \delta]$ (speed and steer).

**Initial condition.** A start set $\mathcal{X}_0$, or a single state $x(0) = x_0$. Here the start is one state: the first waypoint of the circuit, $0.1\,\mathrm{m}$ off the centerline, heading along the path.

**Horizon.** The length $T$ of the interval on which the cost is integrated. Here $T = 1\,\mathrm{s}$.

**Cost.** A running cost $g(x,u)$ and a terminal cost $h(x(T))$,
$$
J = \int_{0}^{T} g\big(x(\tau),\,u(\tau)\big)\,d\tau + h\big(x(T)\big).
$$
Here $g = g_{\mathrm{quad}}(u) + g_{\mathrm{lane}}(x) + g_{\mathrm{obs}}(x)$ of §4, and $h \equiv 0$.

**Input set.** A hard constraint $u(\tau)\in\mathcal{U}$. Here
$$
\mathcal{U} = [0,\, 5]\times[-0.52,\, 0.52]
$$
in $(\mathrm{m/s},\,\mathrm{rad})$: forward speed only, steer within about $\pm 30^{\circ}$.

**State set.** A hard constraint $x(\tau)\in\mathcal{X}$; a plan that leaves $\mathcal{X}$ is infeasible. Here there is no hard $\mathcal{X}$. The lane and the cones are asked through $g$, not as a set.

**Terminal set.** A hard constraint $x(T)\in\mathcal{X}_f$ (a goal region). Here there is none: $x(T)$ is free.

Assembled, the program on this car is
$$
\begin{aligned}
\min_{x(\cdot),\,u(\cdot)}
\quad
&
\int_{0}^{T}
\Big(
g_{\mathrm{quad}}\big(u(\tau)\big)
+
g_{\mathrm{lane}}\big(x(\tau)\big)
+
g_{\mathrm{obs}}\big(x(\tau)\big)
\Big)\,d\tau
\\
\text{s.t.}
\quad
&
\dot x = f(x,u),
\quad
x(0) = x_0,
\quad
u(\tau)\in\mathcal{U}.
\end{aligned}
$$


In [ ]:
problem = PlanningProblem(sys=car, x_start=x0, cost=cost, tf=MPC_HORIZON)
problem


## 6. Trajectory optimization

The program of §5 is infinite-dimensional: the unknowns are functions $x(\cdot)$, $u(\cdot)$ on $[0,T]$. A transcription replaces those functions by a finite decision vector $z$ — the state and the input at $N$ knots — and replaces $\dot x = f(x,u)$ by equality defects between consecutive knots. The result is a nonlinear program
$$
\min_z \; J(z)
\qquad
\text{s.t.}
\quad
c(z) = 0,
\quad
z_{\min}\le z\le z_{\max}.
$$
The equalities $c(z)=0$ are the dynamics and the initial condition. The bounds are the input set $\mathcal{U}$ (and $\mathcal{X}$, when a hard state set is present).

**SLSQP** (sequential least-squares quadratic programming) solves that NLP by repeating three steps:

1. At the current $z$, form a quadratic model of $J$ and a linear model of the constraints.
2. Solve that quadratic program for a step $\Delta z$.
3. Take the step and repeat until the stationarity and constraint residuals are small.

One solve from $x_0$ returns the planned $x(\cdot)$ and $u(\cdot)$ of length $T$.


In [ ]:
planner = TrajectoryOptimizationPlanner(
    problem,
    n_steps=N_STEPS,
    transcription="direct_collocation",
    compile_backend="jax",
    optimizer_method="scipy_slsqp",
    optimizer_options={"maxiter": 40, "ftol": 0.05},
    feasibility_tol=1e-3,
)


In [ ]:
solution = planner.solve()
solution


In [ ]:
planner.plot_solution()


In [ ]:
planner.animate_solution()

## 7. Receding horizon

MPC applies the first sample of that plan, waits $\Delta t$, measures the new state, and **resolves the same program** from there — a new $x(t)$ in the equality constraint, the same $g$, $\mathcal{U}$ and $T$. Only the first input of each new plan is sent to the car.

$$
u(t) = u^*_{t:t+T}(t),
\qquad
\text{then shift $t\leftarrow t+\Delta t$ and solve again.}
$$

`ModelPredictiveController` wraps the planner in that loop. `mpc @ car` closes the loop. The computer ticks every `MPC_DT`; the car integrates in between.


In [ ]:
mpc = ModelPredictiveController(planner, dt_mpc=MPC_DT, warm_start=True, verbose=False)
diagram = mpc @ car


In [ ]:
diagram.plot_diagram()


In [ ]:
result = diagram.compute_trajectory(
    tf=TF, x0_plant=x0, plant_dt_inner=0.01, compile_backend="jax"
)
overlays = mpc_animation_overlays(result, planner, scene=scene, track=track)


In [ ]:
diagram.plot_trajectory()


## 8. Closed-loop lap

The 2-D animation shows the lane, the cones, and the current planned horizon of length $T$.


In [ ]:
car.skin = racecar_skin_2d
diagram.animate(overlays=overlays)


## 3D Animation

Un-comment the following code for 3D animation in browser if running locally

In [ ]:
# car.skin = racecar_skin_3d
# diagram.animate(
#     overlays=overlays,
#     renderer="meshcat",
#     is_3d=True,
#     native=False,
#     html=False,
# )


## 9. Experiments

- Write the program of §5 and name each ingredient on this car.
- Change `V_REF`.
- Move a cone, or add `Sphere((1.0, 2.0), 0.15)` to `scene`.
- Change the hinge weights (`20.0` on the lane, `40.0` on the cones).
- Shorten `MPC_HORIZON`.
